In [12]:
pip install pandas mlflow scikit-learn matplotlib pyarrow mlflow.sklearn os

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement mlflow.sklearn (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\rusru\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for mlflow.sklearn


In [ ]:
import os
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


def main():
    # -----------------------------
    # 1. SET UP MLFLOW
    # -----------------------------
    tracking_dir = os.path.abspath("mlruns")
    tracking_uri = "file:///" + tracking_dir.replace("\\", "/")
    mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_experiment("Spotify_Churn_Logistic_Regression")

    print("Tracking URI:", tracking_uri)
    print()

    # -----------------------------
    # 2. LOAD DATA
    # -----------------------------
    file_name = "../data/processed/spotify_user_behavior_features.parquet"
    df = pd.read_parquet(file_name)

    print("Dataset loaded successfully.")
    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())
    print()

    # -----------------------------
    # 3. BASIC CLEANUP
    # -----------------------------
    target_column = "churn"

    if target_column not in df.columns:
        raise ValueError(f"Target column '{target_column}' not found in dataset.")

    df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")
    df["days_since_signup_start"] = (
        df["signup_date"] - df["signup_date"].min()
    ).dt.days

    drop_columns = ["user_id", "signup_date"]

    df = df.dropna(subset=[target_column])

    X = df.drop(columns=drop_columns + [target_column], errors="ignore")
    y = df[target_column].astype(int)

    print("Target distribution:")
    print(y.value_counts())
    print()

    # -----------------------------
    # 4. IDENTIFY COLUMN TYPES
    # -----------------------------
    numeric_features = X.select_dtypes(
        include=["int64", "int32", "float64", "float32"]
    ).columns.tolist()

    categorical_features = X.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    print("Numeric features:")
    print(numeric_features)
    print()
    print("Categorical features:")
    print(categorical_features)
    print()

    # -----------------------------
    # 5. PREPROCESSING
    # -----------------------------
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )

    # -----------------------------
    # 6. TRAIN / TEST SPLIT
    # -----------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # -----------------------------
    # 7. TRAIN MULTIPLE RUNS
    # -----------------------------
    c_values = [0.01, 0.1, 1.0, 10.0]

    best_accuracy = 0.0
    best_c = None

    for c_val in c_values:
        with mlflow.start_run(run_name=f"logreg_C_{c_val}"):
            model_pipeline = Pipeline(steps=[
                ("preprocessor", preprocessor),
                ("classifier", LogisticRegression(
                    C=c_val,
                    max_iter=2000,
                    random_state=42
                ))
            ])

            # Train
            model_pipeline.fit(X_train, y_train)

            # Predict
            y_pred = model_pipeline.predict(X_test)

            # Evaluate
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall = recall_score(y_test, y_pred, zero_division=0)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            cm = confusion_matrix(y_test, y_pred)
            report = classification_report(y_test, y_pred)

            # Print
            print("=" * 45)
            print(f"Run for C = {c_val}")
            print("Accuracy:", accuracy)
            print("Precision:", precision)
            print("Recall:", recall)
            print("F1 Score:", f1)
            print("Confusion Matrix:")
            print(cm)
            print("Classification Report:")
            print(report)

            # Log params
            mlflow.log_param("model_type", "LogisticRegression")
            mlflow.log_param("dataset_file", file_name)
            mlflow.log_param("target_column", target_column)
            mlflow.log_param("C", c_val)
            mlflow.log_param("max_iter", 2000)
            mlflow.log_param("train_rows", len(X_train))
            mlflow.log_param("test_rows", len(X_test))
            mlflow.log_param("numeric_feature_count", len(numeric_features))
            mlflow.log_param("categorical_feature_count", len(categorical_features))

            # Log metrics
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)

            # Log model artifact
            safe_c_val = str(c_val).replace(".", "_")
            mlflow.sklearn.log_model(
                sk_model=model_pipeline,
                artifact_path="log_reg_model_C_" + safe_c_val
            )

            # Track best
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_c = c_val

    print("=" * 45)
    print("Best C value:", best_c)
    print("Best accuracy:", best_accuracy)
    print()
    print("Done. Open MLflow UI with:")
    print('mlflow ui --backend-store-uri "{}"'.format(tracking_uri))


if __name__ == "__main__":
    main()

Tracking URI: file:///c:/Users/rusru/ML Final Project/ml-design-final-project/notebooks/mlruns

Dataset loaded successfully.
Shape: (50000, 23)
Columns:
['user_id', 'country', 'age', 'signup_date', 'subscription_type', 'churn', 'months_inactive', 'inactive_3_months_flag', 'ad_interaction', 'ad_conversion_to_subscription', 'music_suggestion_rating_1_to_5', 'avg_listening_hours_per_week', 'favorite_genre', 'most_liked_feature', 'desired_future_feature', 'primary_device', 'playlists_created', 'avg_skips_per_day', 'likes_personalization', 'dislikes_suggestions', 'heavy_listener', 'many_playlists', 'new_user']

Target distribution:
churn
0    42109
1     7891
Name: count, dtype: int64

Numeric features:
['age', 'months_inactive', 'inactive_3_months_flag', 'ad_interaction', 'ad_conversion_to_subscription', 'music_suggestion_rating_1_to_5', 'avg_listening_hours_per_week', 'playlists_created', 'avg_skips_per_day', 'likes_personalization', 'dislikes_suggestions', 'heavy_listener', 'many_playlis

2026/04/09 16:24:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 16:24:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run for C = 0.01
Accuracy: 0.9362
Precision: 0.7120938628158845
Recall: 1.0
F1 Score: 0.8318397469688983
Confusion Matrix:
[[7784  638]
 [   0 1578]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.92      0.96      8422
           1       0.71      1.00      0.83      1578

    accuracy                           0.94     10000
   macro avg       0.86      0.96      0.90     10000
weighted avg       0.95      0.94      0.94     10000



2026/04/09 16:24:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 16:24:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run for C = 0.1
Accuracy: 0.9362
Precision: 0.7120938628158845
Recall: 1.0
F1 Score: 0.8318397469688983
Confusion Matrix:
[[7784  638]
 [   0 1578]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.92      0.96      8422
           1       0.71      1.00      0.83      1578

    accuracy                           0.94     10000
   macro avg       0.86      0.96      0.90     10000
weighted avg       0.95      0.94      0.94     10000



2026/04/09 16:24:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 16:24:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run for C = 1.0
Accuracy: 0.9362
Precision: 0.7120938628158845
Recall: 1.0
F1 Score: 0.8318397469688983
Confusion Matrix:
[[7784  638]
 [   0 1578]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.92      0.96      8422
           1       0.71      1.00      0.83      1578

    accuracy                           0.94     10000
   macro avg       0.86      0.96      0.90     10000
weighted avg       0.95      0.94      0.94     10000



2026/04/09 16:24:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 16:24:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run for C = 10.0
Accuracy: 0.9362
Precision: 0.7120938628158845
Recall: 1.0
F1 Score: 0.8318397469688983
Confusion Matrix:
[[7784  638]
 [   0 1578]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.92      0.96      8422
           1       0.71      1.00      0.83      1578

    accuracy                           0.94     10000
   macro avg       0.86      0.96      0.90     10000
weighted avg       0.95      0.94      0.94     10000

Best C value: 0.01
Best accuracy: 0.9362

Done. Open MLflow UI with:
mlflow ui --backend-store-uri file:///c:/Users/rusru/ML Final Project/ml-design-final-project/notebooks/mlruns
